# Transformer DGA Fault Classification — Consolidated Analysis

**A2 Option 2 — Machine Learning System for Transformer DGA Fault Classification**

## Purpose

This notebook consolidates the complete investigation into a single, structured Jupyter Notebook while preserving the four analytical stages:

1. **V1 — Dataset Validity & Initial Signal Analysis**
2. **V2 — DGA Signal Robustness & Tuning Gate**
3. **V2.5 — Local Signal / Target Coherence Diagnostic**
4. **V3 — Final Fixed-Model Error Analysis**

Each version is deliberately formatted as a standalone analysis stage, following the same presentation style used in V1: clear objectives, numbered sections, explanatory markdown, separated code cells, diagnostic outputs, and an explicit conclusion.

> **Important:** V2, V2.5 and V3 do not introduce hyperparameter tuning. They are diagnostic and validation stages designed to determine whether further model development is scientifically justified.

## 1. Dataset location

The four stages use the same company-supplied synthetic DGA dataset.

The notebook first looks for `dummy_dga_dataset.xlsx` in a `Data` folder beside the notebook, then beside the notebook itself, and finally in `/mnt/data` when running in the ChatGPT execution environment.

In [ ]:
from pathlib import Path

DATA_FILENAME = "dummy_dga_dataset.xlsx"

_data_candidates = [
    Path.cwd() / "Data" / DATA_FILENAME,
    Path.cwd() / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]

DATA_PATH = next((p for p in _data_candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Checked:\n"
        + "\n".join(str(p) for p in _data_candidates)
        + "\n\nPlace the dataset in the project's Data folder or beside this notebook."
    )

print("Dataset path:", DATA_PATH)

---
# V1 — Dataset Validity & Initial Signal Analysis

**Purpose**

The first stage determines whether the supplied synthetic DGA dataset contains a meaningful, learnable relationship between DGA measurements and the four-class `fault_by_rogers` target.

The V1 analysis is intentionally conservative. It checks data quality, target balance, leakage, feature distributions, mutual information, baseline models, feature representations, a fixed XGBoost diagnostic, permutation-label controls, and the train-validation gap before deciding whether extensive tuning is justified.

# Transformer DGA Fault Classification — Dataset Validity & Signal Analysis

**A2 Option 2**

## Purpose

This notebook is the first stage of the project. Its purpose is to determine whether the supplied company-provided synthetic DGA dataset contains a meaningful, learnable relationship between DGA measurements and the `fault_by_rogers` target.

We will **not** perform extensive hyperparameter tuning here.

Instead, we will:

1. Load and audit the dataset.
2. Document the dataset structure and target.
3. Identify potential target leakage.
4. Examine whether DGA variables differ across fault classes.
5. Establish a majority-class baseline.
6. Test simple and nonlinear classifiers using stratified cross-validation.
7. Repeat evaluation with permuted labels as a control experiment.
8. Decide whether the dataset provides enough evidence to proceed to the main modelling stage.

> **Confidentiality:** The underlying industrial source data and company identity must not be disclosed. This notebook uses only the synthetic dataset supplied for academic use.

In [ ]:
# 2. Environment setup

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "xgboost": "xgboost",
}

missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Environment ready.")

In [ ]:
# 3. Imports and reproducibility

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    make_scorer,
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)

## 4. Load the dataset

### Expected file

The notebook assumes the uploaded file is:

`dummy_dga_dataset.xlsx`

If using Google Colab, upload the file to the Colab session or change `DATA_PATH` to the location of the uploaded file.

In [ ]:
# 4. Load the dataset

df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

In [ ]:
# 5. Basic dataset audit

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())

## 5.1 Target distribution

The main target is `fault_by_rogers`.

We will inspect the class distribution before modelling because accuracy alone can be misleading when class sizes differ.

In [ ]:
TARGET = "fault_by_rogers"

if TARGET not in df.columns:
    raise KeyError(
        f"'{TARGET}' was not found. Available columns are:\n{df.columns.tolist()}"
    )

target_counts = df[TARGET].value_counts(dropna=False)
target_pct = df[TARGET].value_counts(normalize=True, dropna=False).mul(100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct
})

display(target_summary)

plt.figure(figsize=(9, 5))
sns.countplot(data=df, x=TARGET, order=target_counts.index)
plt.title("Target Class Distribution")
plt.xlabel("Fault Class")
plt.ylabel("Number of observations")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 6. Column classification

We will explicitly define which variables are allowed to become model inputs.

### Candidate raw DGA features

- H2
- C2H2
- C2H4
- C2H6
- CH4
- CO
- CO2

### Candidate engineered features

- R_C2H2_C2H4
- R_CH4_H2
- R_C2H4_C2H6

### Variables excluded from predictive modelling

- target
- `abnormal`
- `status`
- `triggers`
- identifiers
- asset identifiers
- timestamps, unless a later experiment provides a defensible reason to use them

This conservative feature definition prevents the first experiment from accidentally learning from variables that already encode the target or the abnormality rule.

In [ ]:
RAW_DGA = [
    "H2",
    "C2H2",
    "C2H4",
    "C2H6",
    "CH4",
    "CO",
    "CO2",
]

RATIO_FEATURES = [
    "R_C2H2_C2H4",
    "R_CH4_H2",
    "R_C2H4_C2H6",
]

required_features = RAW_DGA + RATIO_FEATURES
missing_features = [c for c in required_features if c not in df.columns]

print("Missing expected DGA features:", missing_features)

if missing_features:
    print("\nAvailable columns:")
    print(df.columns.tolist())
else:
    print("\nAll expected DGA features are present.")

FEATURE_SET_RAW = RAW_DGA
FEATURE_SET_RAW_RATIOS = RAW_DGA + RATIO_FEATURES
FEATURE_SET_RATIOS = RATIO_FEATURES

print("\nRaw DGA:", FEATURE_SET_RAW)
print("Raw + ratios:", FEATURE_SET_RAW_RATIOS)
print("Ratios only:", FEATURE_SET_RATIOS)

# 7. Leakage investigation

Before training, inspect variables that may already encode the abnormality decision.

The dataset contains:

- `abnormal`
- `status`
- `triggers`

These are **not model inputs**.

We will inspect their relationships with the DGA measurements and the target to document why they are excluded.

In [ ]:
leakage_candidates = [c for c in ["abnormal", "status", "triggers"] if c in df.columns]

for col in leakage_candidates:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

In [ ]:
# Check whether abnormality is exactly determined by C2H4 > 60.
if "abnormal" in df.columns and "C2H4" in df.columns:
    rule = df["C2H4"] > 60
    abnormal_bool = df["abnormal"].astype(bool)

    print("Rows where (C2H4 > 60) disagrees with abnormal:", (rule != abnormal_bool).sum())
    print("Rows where C2H4 > 60:", rule.sum())
    print("Rows marked abnormal:", abnormal_bool.sum())

In [ ]:
# Cross-tabs for existing labels/status/trigger relationships
for col in ["abnormal", "status", "triggers"]:
    if col in df.columns:
        print(f"\nCross-tab: {col} vs {TARGET}")
        display(pd.crosstab(df[col], df[TARGET], margins=True))

### Leakage decision

If a variable directly encodes an existing rule or the target, it is excluded from predictive modelling.

This is important because a model that receives `status`, `abnormal`, or `triggers` would not be solving the intended deployment problem:

> **DGA measurements → predicted transformer condition**

# 8. DGA feature distributions by fault class

The first evidence of learnability should come from the feature distributions.

If the same DGA measurement has substantially different distributions across fault classes, that provides a reason to expect a classifier may learn useful boundaries.

If the distributions strongly overlap, classification may be intrinsically difficult.

In [ ]:
for feature in RAW_DGA:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x=TARGET, y=feature)
    plt.title(f"{feature} by Fault Class")
    plt.xlabel("Fault Class")
    plt.ylabel(feature)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# Class-wise summary statistics for raw DGA features
class_means = df.groupby(TARGET)[RAW_DGA].mean().T
class_medians = df.groupby(TARGET)[RAW_DGA].median().T

print("Class-wise means:")
display(class_means)

print("Class-wise medians:")
display(class_medians)

## 8.1 Correlation structure

Correlation does not prove predictive usefulness, but it can reveal redundant variables and relationships between the DGA measurements.

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[RAW_DGA + RATIO_FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of DGA Features")
plt.tight_layout()
plt.show()

# 9. Mutual information screening

Mutual information provides a non-linear dependence measure between each feature and the target.

It is **not** a final feature-selection method here.

Its purpose is diagnostic:

> Is there measurable dependence between individual DGA variables and the fault labels?

In [ ]:
X_mi = df[FEATURE_SET_RAW_RATIOS].copy()
y_mi = df[TARGET].copy()

# Encode target only for the MI calculation.
le = LabelEncoder()
y_encoded = le.fit_transform(y_mi)

mi_values = mutual_info_classif(
    X_mi,
    y_encoded,
    random_state=RANDOM_STATE
)

mi_summary = (
    pd.DataFrame({
        "feature": X_mi.columns,
        "mutual_information": mi_values
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mi_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=mi_summary, x="mutual_information", y="feature")
plt.title("Mutual Information Between DGA Features and Fault Class")
plt.xlabel("Mutual information")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# 10. Baseline evaluation

We now establish the minimum performance that a useful classifier should beat.

### Baseline

A `DummyClassifier` always predicts the most frequent class.

### Main diagnostic models

1. Multiclass Logistic Regression — relatively simple model.
2. Extra Trees — nonlinear tree ensemble.
3. XGBoost — nonlinear gradient-boosted tree model.

At this stage we are testing **learnability**, not optimising the final model.

In [ ]:
X = df[FEATURE_SET_RAW_RATIOS].copy()
y = df[TARGET].copy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": make_scorer(f1_score, average="macro"),
    "weighted_f1": make_scorer(f1_score, average="weighted"),
}

models = {
    "Majority baseline": Pipeline([
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),

    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ]),
}

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    results.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_macro_f1_mean": scores["test_macro_f1"].mean(),
        "cv_macro_f1_std": scores["test_macro_f1"].std(),
        "cv_weighted_f1_mean": scores["test_weighted_f1"].mean(),
        "cv_weighted_f1_std": scores["test_weighted_f1"].std(),
        "cv_balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "cv_balanced_accuracy_std": scores["test_balanced_accuracy"].std(),
        "train_macro_f1_mean": scores["train_macro_f1"].mean(),
    })

baseline_results = pd.DataFrame(results).sort_values(
    "cv_macro_f1_mean",
    ascending=False
)

display(baseline_results)

# 11. Feature-representation experiment

The broader project asks whether engineered DGA ratios add useful information.

We will therefore compare:

- Raw DGA only
- Raw DGA + ratios
- Ratios only

using the same model and the same cross-validation strategy.

This is a controlled experiment: the validation procedure and model family remain constant while the representation changes.

In [ ]:
representation_sets = {
    "Raw DGA": FEATURE_SET_RAW,
    "Raw + ratios": FEATURE_SET_RAW_RATIOS,
    "Ratios only": FEATURE_SET_RATIOS,
}

representation_results = []

for feature_name, features in representation_sets.items():
    X_rep = df[features]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ])

    scores = cross_validate(
        model,
        X_rep,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    representation_results.append({
        "feature_representation": feature_name,
        "macro_f1_mean": scores["test_macro_f1"].mean(),
        "macro_f1_std": scores["test_macro_f1"].std(),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
    })

representation_results = pd.DataFrame(representation_results)
display(representation_results)

# 12. XGBoost signal check

XGBoost is the main candidate model for the eventual project, but here it is used only as another test of whether nonlinear relationships can be learned.

The hyperparameters are deliberately conservative rather than extensively tuned.

In [ ]:
# Encode target labels for XGBoost.
target_encoder = LabelEncoder()
y_xgb = target_encoder.fit_transform(y)

xgb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=len(target_encoder.classes_),
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    ))
])

xgb_scores = cross_validate(
    xgb_model,
    X,
    y_xgb,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

xgb_summary = pd.DataFrame([{
    "model": "XGBoost",
    "cv_accuracy_mean": xgb_scores["test_accuracy"].mean(),
    "cv_accuracy_std": xgb_scores["test_accuracy"].std(),
    "cv_macro_f1_mean": xgb_scores["test_macro_f1"].mean(),
    "cv_macro_f1_std": xgb_scores["test_macro_f1"].std(),
    "cv_weighted_f1_mean": xgb_scores["test_weighted_f1"].mean(),
    "cv_weighted_f1_std": xgb_scores["test_weighted_f1"].std(),
    "cv_balanced_accuracy_mean": xgb_scores["test_balanced_accuracy"].mean(),
    "cv_balanced_accuracy_std": xgb_scores["test_balanced_accuracy"].std(),
    "train_macro_f1_mean": xgb_scores["train_macro_f1"].mean(),
}])

display(xgb_summary)

# 13. Permutation-label control experiment

This is a key validity check.

We keep the DGA features unchanged but randomly permute the target labels.

If the real-label model performs substantially better than models trained on permuted labels, that supports the conclusion that the DGA variables contain information related to the target.

If real-label and permuted-label performance are similar, there is little evidence of learnable signal.

This is a **control experiment**, not a replacement for proper validation.

In [ ]:
N_PERMUTATIONS = 30

permutation_scores = []

for seed in range(RANDOM_STATE, RANDOM_STATE + N_PERMUTATIONS):
    rng = np.random.default_rng(seed)
    y_perm = rng.permutation(y_xgb)

    scores = cross_validate(
        xgb_model,
        X,
        y_perm,
        cv=cv,
        scoring=make_scorer(f1_score, average="macro"),
        n_jobs=-1
    )

    permutation_scores.append(scores["test_score"].mean())

permutation_scores = np.array(permutation_scores)

real_macro_f1 = xgb_summary.loc[0, "cv_macro_f1_mean"]

print(f"Real-label mean macro-F1: {real_macro_f1:.4f}")
print(f"Permuted-label mean macro-F1: {permutation_scores.mean():.4f}")
print(f"Permuted-label std: {permutation_scores.std():.4f}")
print(f"Permuted-label 95th percentile: {np.percentile(permutation_scores, 95):.4f}")

plt.figure(figsize=(9, 5))
sns.histplot(permutation_scores, bins=10, kde=True)
plt.axvline(real_macro_f1, linestyle="--", linewidth=2, label="Real-label score")
plt.axvline(np.percentile(permutation_scores, 95), linestyle=":", linewidth=2,
            label="95th percentile of permuted labels")
plt.title("Permutation-Label Control")
plt.xlabel("Cross-validated macro-F1")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# 14. Train-vs-validation gap

A model can achieve a high training score simply by memorising the training data.

We therefore compare training and validation macro-F1.

A large gap is evidence of overfitting and will affect how we interpret the model.

In [ ]:
gap_table = pd.DataFrame([{
    "model": "XGBoost",
    "train_macro_f1": xgb_scores["train_macro_f1"].mean(),
    "validation_macro_f1": xgb_scores["test_macro_f1"].mean(),
    "generalisation_gap": (
        xgb_scores["train_macro_f1"].mean()
        - xgb_scores["test_macro_f1"].mean()
    )
}])

display(gap_table)

# 15. Preliminary GO / NO-GO decision

The decision is deliberately evidence-based.

### GO

Proceed to the full project if:

1. The real-label model performs meaningfully above the majority baseline, **and**
2. Real-label performance is clearly above the permutation-label control, **and**
3. The model does not rely only on extreme overfitting.

### INVESTIGATE

If there is some evidence of signal but the result is weak or unstable, investigate:

- class overlap
- feature representation
- validation design
- synthetic-data generation
- target quality

### NO-GO / DATA REVIEW

If real-label performance is approximately the same as the permutation control, do not spend time on extensive hyperparameter tuning.

Instead, review the synthetic-data generation process with the data provider.

> This is not a failure of the ML project. Demonstrating that the available representation does not contain sufficient predictive information is itself an important validity finding. The problem would be claiming strong predictive performance when the evidence does not support it.

In [ ]:
# 16. Automated preliminary decision

majority_f1 = baseline_results.loc[
    baseline_results["model"] == "Majority baseline",
    "cv_macro_f1_mean"
].iloc[0]

real_score = float(real_macro_f1)
perm_mean = float(permutation_scores.mean())
perm_p95 = float(np.percentile(permutation_scores, 95))
train_score = float(xgb_scores["train_macro_f1"].mean())

print("========== PRELIMINARY DATASET VALIDITY DECISION ==========")
print(f"Majority baseline macro-F1 : {majority_f1:.4f}")
print(f"Real-label XGBoost macro-F1: {real_score:.4f}")
print(f"Permutation mean macro-F1  : {perm_mean:.4f}")
print(f"Permutation 95th percentile: {perm_p95:.4f}")
print(f"XGBoost training macro-F1  : {train_score:.4f}")
print(f"Generalisation gap         : {train_score - real_score:.4f}")
print()

beats_baseline = real_score > majority_f1
beats_permutation = real_score > perm_p95

if beats_baseline and beats_permutation:
    print("DECISION: GO — evidence of learnable signal.")
elif beats_baseline or beats_permutation:
    print("DECISION: INVESTIGATE — some evidence exists, but it is not yet strong enough for a confident GO.")
else:
    print("DECISION: DATA REVIEW — current evidence does not demonstrate useful learnable signal.")

---
# V2 — DGA Signal Robustness & Tuning Gate

**Purpose**

V2 asks whether the preliminary V1 signal survives stronger validation and robustness checks.

The analysis examines target relationships, class separability, asset effects, temporal structure, repeated stratified cross-validation, a permutation-label control, alternative DGA representations, leave-one-asset-out validation, chronological validation, and a conservative all-pass gate for deciding whether tuning would be justified.

The structure below follows the same style as V1: environment setup → imports and reproducibility → dataset loading → numbered analytical sections → explicit conclusion.

In [ ]:
# 2. Environment setup

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "xgboost": "xgboost",
}

missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Environment ready.")

In [ ]:
# 3. Imports and reproducibility

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, kruskal

from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42
N_PERMUTATIONS = 100
np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)
print("Number of label permutations:", N_PERMUTATIONS)

## 4. Load the dataset

The same `dummy_dga_dataset.xlsx` used in V1 is loaded so that the validation stages operate on the identical observations and target definition.

In [ ]:
df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 5. Target-generation and label relationship checks

The diagnostic fields `status`, `abnormal`, and `triggers` remain excluded from predictive modelling. This section checks how those fields relate to the target and evaluates univariate DGA dependence using Kruskal–Wallis statistics, effect size, and mutual information.

These tests are descriptive; they cannot reveal a hidden target-generation rule that is not represented in the supplied dataset.

In [ ]:
# 5. Target relation checks: diagnostic fields remain excluded from models.

T, A, DT = "fault_by_rogers", "asset", "ts_H2"
raw = ["H2", "C2H2", "C2H4", "C2H6", "CH4", "CO", "CO2"]
ratios = ["R_C2H2_C2H4", "R_CH4_H2", "R_C2H4_C2H6"]
features = raw + ratios

df[DT] = pd.to_datetime(df[DT])
classes = sorted(df[T].unique())
y = LabelEncoder().fit_transform(df[T])
labels = np.arange(len(classes))
X = df[features]

print(f"Rows={len(df)}; time range: {df[DT].min()} to {df[DT].max()}")
display(
    pd.DataFrame({
        "count": df[T].value_counts(),
        "proportion": df[T].value_counts(normalize=True)
    }).sort_index()
)

print("\nTarget-generation / label relationship checks")
for c in ["status", "abnormal", "triggers"]:
    print(f"\n{c} by target")
    display(pd.crosstab(df[c].fillna("<missing>"), df[T], normalize="index").round(3))

mi = mutual_info_classif(X, y, random_state=RANDOM_STATE)
out = []

for c in features:
    h, p = kruskal(*[g[c].dropna() for _, g in df.groupby(T)])
    out.append([
        c,
        h,
        p,
        max(0, (h - len(classes) + 1) / (len(df) - len(classes)))
    ])

tbl = pd.DataFrame(
    out,
    columns=["feature", "Kruskal_H", "p_unadjusted", "epsilon_squared"]
)
tbl["mutual_information"] = mi

display(tbl.sort_values("epsilon_squared", ascending=False).round(4))

print(
    "The data cannot prove its hidden generator. Provider question: "
    "which DGA fields/rules/thresholds/transformations/randomisation generated "
    "fault_by_rogers, and were labels independent of DGA?"
)

## 6. Class separability and feature distributions

The next diagnostic checks whether the four target classes occupy visibly different regions of the supplied DGA feature space.

The plots are exploratory diagnostics rather than model-selection steps.

In [ ]:
# 6. Separability plots and PCA, never a model.

fig, ax = plt.subplots(2, 4, figsize=(18, 9))

for z, c in zip(ax.flat, raw):
    sns.boxplot(
        data=df,
        x=T,
        y=c,
        order=classes,
        showfliers=False,
        ax=z
    )
    z.set_title(c + " by class")
    z.tick_params(axis="x", rotation=25)

ax.flat[-1].axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(2, 2, figsize=(13, 11))

for z, (x1, x2) in zip(
    ax.flat,
    [("H2", "CH4"), ("H2", "C2H2"), ("C2H4", "C2H6"), ("C2H2", "C2H4")]
):
    sns.scatterplot(
        data=df,
        x=x1,
        y=x2,
        hue=T,
        hue_order=classes,
        alpha=.75,
        ax=z
    )
    z.set_title(f"{x2} vs {x1}")

plt.tight_layout()
plt.show()

xx = Pipeline([
    ("i", SimpleImputer(strategy="median")),
    ("s", StandardScaler())
]).fit_transform(X)

pc = PCA(n_components=2, random_state=42)
p = pc.fit_transform(xx)

print(f"PCA PC1+PC2 variance explained: {pc.explained_variance_ratio_.sum():.1%}")

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pd.DataFrame({"PC1": p[:, 0], "PC2": p[:, 1], T: df[T]}),
    x="PC1",
    y="PC2",
    hue=T,
    hue_order=classes,
    alpha=.8
)
plt.title("PCA visual diagnostic only")
plt.tight_layout()
plt.show()

## 7. Asset-target relationship

`asset` is treated as a grouping variable rather than a predictive feature. This section checks whether class composition differs substantially between assets.

This is important because a model that relies on asset identity would not demonstrate general DGA-to-fault classification.

In [ ]:
# 7. Asset diagnostic; asset is never in X.

print("Asset-target relationship")

ac = pd.crosstab(df[A], df[T]).reindex(columns=classes, fill_value=0)
chi, pv, _, _ = chi2_contingency(ac)
v = np.sqrt(
    (chi / ac.to_numpy().sum())
    / min(ac.shape[0] - 1, ac.shape[1] - 1)
)

display(ac)
print(f"chi-square p={pv:.4g}; Cramer's V={v:.3f}")

ac.div(ac.sum(axis=1), axis=0).plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    colormap="tab10"
)
plt.ylabel("within-asset class proportion")
plt.tight_layout()
plt.show()

## 8. Temporal structure

The timestamp is also excluded from `X`. The diagnostic below checks the observation period, daily sampling structure, daily class composition, and whether chronologically adjacent observations tend to come from the same asset.

In [ ]:
# 8. Temporal diagnostics; timestamp is never in X.

print("Temporal structure")
display(df.groupby(A)[DT].agg(["count", "min", "max"]))

q = df[[DT, A, T]].sort_values(DT).copy()
q["day"] = q[DT].dt.date

fig, ax = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

q.groupby("day").size().plot(ax=ax[0], marker="o")
ax[0].set_title("Rows by day")

pd.crosstab(
    q["day"],
    q[T],
    normalize="index"
).reindex(columns=classes, fill_value=0).plot(
    kind="bar",
    stacked=True,
    ax=ax[1],
    colormap="tab10"
)

ax[1].set_title("Class composition by day")
plt.tight_layout()
plt.show()

a = q[A].to_numpy()
print(
    f"Chronologically adjacent rows from same asset: "
    f"{(a[1:] == a[:-1]).mean():.1%}"
)

## 9. Fixed XGBoost configuration

The same conservative XGBoost configuration is retained rather than tuned.

The purpose of V2 is to test whether the signal is robust enough to justify tuning later, not to optimise the model.

In [ ]:
# 9. Exact V1 XGBoost configuration; intentionally no tuning.

def mod():
    return Pipeline([
        ("i", SimpleImputer(strategy="median")),
        ("m", XGBClassifier(
            objective="multi:softprob",
            num_class=len(classes),
            n_estimators=150,
            max_depth=3,
            learning_rate=.05,
            subsample=.9,
            colsample_bytree=.9,
            reg_lambda=1.,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=1,
            tree_method="hist"
        ))
    ])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "f1": "f1_macro",
    "accuracy": "accuracy",
    "balanced": "balanced_accuracy"
}

print("Fixed model and repeated cross-validation configured.")

## 10. Repeated cross-validation and train-validation gap

A repeated 5-fold stratified design provides a more stable estimate than a single train/test split.

The training score is also retained to quantify whether the model is memorising the supplied observations.

In [ ]:
# 10. Repeated CV, including train-validation gap.

print("\nRepeated stratified CV: 5 folds x 5 repeats")

real = cross_validate(
    mod(),
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

base = cross_validate(
    DummyClassifier(strategy="most_frequent"),
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

def sm(n, z):
    return [
        n,
        z["test_f1"].mean(),
        z["test_f1"].std(ddof=1),
        z["train_f1"].mean(),
        z["train_f1"].mean() - z["test_f1"].mean(),
        z["test_accuracy"].mean(),
        z["test_balanced"].mean()
    ]

cvt = pd.DataFrame(
    [
        sm("Fixed XGBoost", real),
        sm("Majority baseline", base)
    ],
    columns=[
        "model",
        "validation_macro_f1",
        "fold_sd",
        "train_macro_f1",
        "gap",
        "accuracy",
        "balanced_accuracy"
    ]
)

display(cvt.round(4))

plt.figure(figsize=(10, 4))
sns.stripplot(
    data=pd.DataFrame({
        "validation": real["test_f1"],
        "training": real["train_f1"]
    }).melt(),
    x="variable",
    y="value",
    jitter=.18
)
plt.title("25 repeated-CV macro-F1 scores")
plt.tight_layout()
plt.show()

## 11. Permutation-label control experiment

The same repeated cross-validation design is repeated after randomly permuting the target labels.

The purpose is to ask whether the observed score is clearly separated from the score obtainable when the relationship between DGA measurements and labels is destroyed.

In [ ]:
# 11. Repeated exact same CV under permuted labels.

print("\nRepeated permutation-label control: 100 null experiments")

pm = []

for seed in range(42, 142):
    pm.append(
        cross_validate(
            mod(),
            X,
            np.random.default_rng(seed).permutation(y),
            cv=cv,
            scoring="f1_macro",
            n_jobs=-1
        )["test_score"].mean()
    )

pm = np.asarray(pm)
rm = real["test_f1"].mean()
p95 = np.quantile(pm, .95)
ep = (1 + (pm >= rm).sum()) / 101

display(
    pd.DataFrame([[
        rm,
        pm.mean(),
        pm.std(ddof=1),
        p95,
        ep
    ]], columns=[
        "real_mean",
        "null_mean",
        "null_sd",
        "null_95th_pct",
        "empirical_one_sided_p"
    ]).round(4)
)

plt.figure(figsize=(10, 5))
sns.histplot(pm, bins=15, kde=True)
plt.axvline(rm, color="crimson", ls="--", label="real")
plt.axvline(p95, color="black", ls=":", label="null 95th")
plt.legend()
plt.title("Repeated permutation control")
plt.tight_layout()
plt.show()

## 12. Robustness checks

Three additional checks are performed:

- **Feature representation:** raw DGA, raw DGA + ratios, and ratios only.
- **Leave-one-asset-out:** trains on two assets and evaluates on the third.
- **Chronological 70/30 holdout:** evaluates on the latest observations after training on earlier observations.

These checks help distinguish an apparent in-sample pattern from a robust classification signal.

In [ ]:
# 12. Representation, leave-one-asset-out and chronological checks.

rr = []

for n, c in {
    "Raw DGA": raw,
    "Raw + ratios": features,
    "Ratios only": ratios
}.items():
    z = cross_validate(
        mod(),
        df[c],
        y,
        cv=cv,
        scoring="f1_macro",
        return_train_score=True,
        n_jobs=-1
    )
    rr.append([
        n,
        z["test_score"].mean(),
        z["train_score"].mean() - z["test_score"].mean()
    ])

rt = pd.DataFrame(
    rr,
    columns=["representation", "repeated_cv_macro_f1", "gap"]
).sort_values("repeated_cv_macro_f1", ascending=False)

display(rt.round(4))

ar = []

for hold in sorted(df[A].unique()):
    tr = df[A].ne(hold)
    te = ~tr

    m = mod().fit(X[tr], y[tr])
    d = DummyClassifier(strategy="most_frequent").fit(X[tr], y[tr])

    ar.append([
        hold,
        tr.sum(),
        te.sum(),
        f1_score(
            y[tr],
            m.predict(X[tr]),
            average="macro",
            labels=labels,
            zero_division=0
        ),
        f1_score(
            y[te],
            m.predict(X[te]),
            average="macro",
            labels=labels,
            zero_division=0
        ),
        f1_score(
            y[te],
            d.predict(X[te]),
            average="macro",
            labels=labels,
            zero_division=0
        )
    ])

at = pd.DataFrame(
    ar,
    columns=[
        "held_out_asset",
        "train_n",
        "test_n",
        "train_macro_f1",
        "test_macro_f1",
        "majority_test_macro_f1"
    ]
)

print("Leave-one-asset-out")
display(at.round(4))

o = df.sort_values(DT).index.to_numpy()
k = int(.7 * len(o))
tr, te = o[:k], o[k:]

m = mod().fit(X.loc[tr], y[tr])
d = DummyClassifier(strategy="most_frequent").fit(X.loc[tr], y[tr])

ct = pd.DataFrame([[
    len(tr),
    len(te),
    f1_score(
        y[tr],
        m.predict(X.loc[tr]),
        average="macro",
        labels=labels,
        zero_division=0
    ),
    f1_score(
        y[te],
        m.predict(X.loc[te]),
        average="macro",
        labels=labels,
        zero_division=0
    ),
    f1_score(
        y[te],
        d.predict(X.loc[te]),
        average="macro",
        labels=labels,
        zero_division=0
    ),
    accuracy_score(y[te], m.predict(X.loc[te])),
    balanced_accuracy_score(y[te], m.predict(X.loc[te]))
]], columns=[
    "train_n",
    "test_n",
    "train_macro_f1",
    "test_macro_f1",
    "majority_test_macro_f1",
    "test_accuracy",
    "test_balanced_accuracy"
])

print("Chronological 70/30 holdout")
display(ct.round(4))

## 13. Conservative GO / NO-GO decision

The tuning gate is deliberately strict. All conditions must pass before isolated XGBoost tuning is considered scientifically justified.

The conditions check:

1. Real performance exceeds the permutation control.
2. Real performance meaningfully exceeds the majority baseline.
3. At least one representation exceeds the permutation threshold.
4. The train-validation gap is not substantial.
5. Asset and chronological checks beat their respective majority baselines.

In [ ]:
# 13. Conservative all-pass tuning gate.

bm = base["test_f1"].mean()
gap = real["train_f1"].mean() - rm
xa = at.test_macro_f1.mean()
ba = at.majority_test_macro_f1.mean()

gate = pd.DataFrame([
    [
        "Real exceeds permutation",
        rm > p95 and ep < .05,
        f"real={rm:.4f}; null p95={p95:.4f}; p={ep:.4f}"
    ],
    [
        "Meaningfully exceeds majority",
        rm - bm >= .05,
        f"margin={rm - bm:.4f}; threshold=.0500"
    ],
    [
        "Some representation exceeds null",
        rt.iloc[0].repeated_cv_macro_f1 > p95,
        f"best={rt.iloc[0].representation}: {rt.iloc[0].repeated_cv_macro_f1:.4f}"
    ],
    [
        "No substantial memorisation",
        gap <= .20,
        f"gap={gap:.4f}; threshold <=.2000"
    ],
    [
        "Asset and time robustness beat baselines",
        xa > ba and ct.loc[0, "test_macro_f1"] > ct.loc[0, "majority_test_macro_f1"],
        f"asset={xa:.4f}/{ba:.4f}; "
        f"time={ct.loc[0, 'test_macro_f1']:.4f}/{ct.loc[0, 'majority_test_macro_f1']:.4f}"
    ]
], columns=["condition", "pass", "evidence"])

print("\nXGBoost tuning gate — all conditions must pass")
display(gate)

if gate["pass"].all():
    verdict = "GO — isolated/nested XGBoost tuning is justified in V3."
elif rm > pm.mean() and rm > bm:
    verdict = (
        "INVESTIGATE — weak/unstable evidence; review target construction, "
        "overlap, asset and temporal effects before tuning."
    )
else:
    verdict = (
        "NO-GO / DATA–TARGET REVIEW — no reproducible DGA to fault-class "
        "signal demonstrated; tuning is not scientifically justified."
    )

print("=" * 72)
print("V2 CONCLUSION")
print(verdict)

---
# V2.5 — Local Signal / Target Coherence Diagnostic

**Purpose**

V2.5 investigates whether observations that are close in the supplied DGA feature space also tend to have the same `fault_by_rogers` label.

This stage is deliberately a diagnostic rather than a tuned KNN model. It tests nearest-neighbour label purity against a permutation control, compares same-class and different-class distances, measures pairwise class separation, and records concrete close cross-label examples.

The structure below follows the same style as V1: environment setup → imports and reproducibility → dataset loading → numbered analytical sections → explicit conclusion.

In [ ]:
# 2. Environment setup

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
}

missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Environment ready.")

In [ ]:
# 3. Imports and reproducibility

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import mannwhitneyu

from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, LabelEncoder

RANDOM_STATE = 42
N_PERMUTATIONS = 100
np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)
print("Number of label permutations:", N_PERMUTATIONS)

## 4. Load the dataset

The same `dummy_dga_dataset.xlsx` used in V1 is loaded so that the validation stages operate on the identical observations and target definition.

In [ ]:
df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 5. Standardised DGA feature space

The supplied DGA features are median-imputed and standardised only for distance-based diagnostics.

No observations or target labels are modified.

In [ ]:
T = "fault_by_rogers"
raw = ["H2", "C2H2", "C2H4", "C2H6", "CH4", "CO", "CO2"]
ratios = ["R_C2H2_C2H4", "R_CH4_H2", "R_C2H4_C2H6"]
features = raw + ratios

classes = sorted(df[T].unique())
y = LabelEncoder().fit_transform(df[T])
labeler = LabelEncoder().fit(df[T])

X = StandardScaler().fit_transform(
    SimpleImputer(strategy="median").fit_transform(df[features])
)

print(
    f"V2.5 local-signal diagnostic: {len(df)} observations, "
    f"{len(features)} supplied DGA features."
)
print(
    "Features are median-imputed then standardised only so distance is not "
    "dominated by CO/CO2 scale. No data values are changed."
)

display(
    pd.DataFrame({
        "count": df[T].value_counts(),
        "proportion": df[T].value_counts(normalize=True)
    }).sort_index()
)

## 6. Nearest-neighbour local label consistency

For every observation, the same fixed set of nearest neighbours is identified.

The observed neighbour-label purity is then compared with 100 label permutations. This preserves the feature geometry and class balance while removing the actual relationship between DGA measurements and target labels.

In [ ]:
# 6. Find the same fixed ten nearest neighbours for every observation, excluding itself.

nn = NearestNeighbors(n_neighbors=11, metric="euclidean").fit(X)
dist, ind = nn.kneighbors(X)
dist, ind = dist[:, 1:], ind[:, 1:]

neighbour_labels = y[ind]

def purity(labels):
    return {
        f"{k}-neighbour purity":
            (neighbour_labels[:, :k] == labels[:, None]).mean()
        for k in (1, 3, 5, 10)
    }

observed = purity(y)

null = {k: [] for k in observed}

for seed in range(RANDOM_STATE, RANDOM_STATE + N_PERMUTATIONS):
    p = np.random.default_rng(seed).permutation(y)
    for k, v in purity(p).items():
        null[k].append(v)

rows = []

for k, v in observed.items():
    a = np.asarray(null[k])
    rows.append([
        k,
        v,
        a.mean(),
        a.std(ddof=1),
        np.quantile(a, .95),
        (1 + (a >= v).sum()) / (N_PERMUTATIONS + 1),
        v - a.mean()
    ])

purity_table = pd.DataFrame(
    rows,
    columns=[
        "metric",
        "observed",
        "permutation_mean",
        "permutation_sd",
        "permutation_95th_pct",
        "empirical_one_sided_p",
        "observed_minus_null_mean"
    ]
)

print(
    "\nLocal label consistency: are nearest DGA neighbours assigned "
    "the same target more often than chance?"
)
display(purity_table.round(4))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=purity_table.melt(
        id_vars="metric",
        value_vars=["observed", "permutation_mean"],
        var_name="source",
        value_name="purity"
    ),
    x="metric",
    y="purity",
    hue="source"
)
plt.title("Observed nearest-neighbour label purity versus permuted labels")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Class-specific local purity

The overall local-purity result can hide differences between fault classes.

This diagnostic therefore reports how frequently neighbours of each class share the same label.

In [ ]:
# 7. Class-specific neighbour purity.

class_rows = []

for ci, c in enumerate(classes):
    mask = y == ci

    for k in (1, 3, 5, 10):
        class_rows.append([
            c,
            k,
            (neighbour_labels[mask, :k] == ci).mean(),
            mask.sum()
        ])

class_purity = pd.DataFrame(
    class_rows,
    columns=["class", "k", "same_class_neighbour_purity", "n"]
)

print("\nClass-specific local purity")
display(
    class_purity.pivot(
        index="class",
        columns="k",
        values="same_class_neighbour_purity"
    ).round(3)
)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=class_purity,
    x="k",
    y="same_class_neighbour_purity",
    hue="class",
    marker="o"
)
plt.title("Local consistency by fault class")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 8. Same-class versus different-class DGA distances

If the target labels reflect meaningful structure in the supplied DGA measurements, same-class observations should tend to be closer together than observations with different labels.

The comparison is descriptive and does not constitute a predictive model.

In [ ]:
# 8. Same-vs-different class distances and close conflicting pairs.

D = squareform(pdist(X, metric="euclidean"))
iu = np.triu_indices(len(df), k=1)

pair_distance = D[iu]
same = y[iu[0]] == y[iu[1]]

same_d = pair_distance[same]
diff_d = pair_distance[~same]

u, p = mannwhitneyu(
    same_d,
    diff_d,
    alternative="less"
)

distance_summary = pd.DataFrame([
    [
        "same class",
        len(same_d),
        same_d.mean(),
        np.median(same_d),
        np.quantile(same_d, .05),
        np.quantile(same_d, .95)
    ],
    [
        "different class",
        len(diff_d),
        diff_d.mean(),
        np.median(diff_d),
        np.quantile(diff_d, .05),
        np.quantile(diff_d, .95)
    ]
], columns=[
    "pair_type",
    "n_pairs",
    "mean_distance",
    "median_distance",
    "p05",
    "p95"
])

print("\nStandardised DGA distance: same-label versus different-label observation pairs")
display(distance_summary.round(4))

print(
    f"One-sided Mann–Whitney test, same-class distances smaller: "
    f"p={p:.4g}. This is descriptive evidence, not a predictive-model result."
)

plt.figure(figsize=(10, 5))
sns.kdeplot(same_d, label="same class", fill=True)
sns.kdeplot(diff_d, label="different class", fill=True)
plt.legend()
plt.title("Distance overlap: same-class vs different-class pairs")
plt.xlabel("standardised Euclidean DGA distance")
plt.tight_layout()
plt.show()

## 9. Pairwise class separation

For every pair of classes, the distance between their standardised centroids is compared with pooled within-class feature variation.

Small standardised separation indicates substantial overlap in the supplied DGA representation.

In [ ]:
# 9. Each class pair: standardised centroid separation relative to pooled within-class spread.

pair_rows = []

for i, c1 in enumerate(classes):
    for j, c2 in enumerate(classes):
        if j <= i:
            continue

        a, b = X[y == i], X[y == j]
        centre = np.linalg.norm(a.mean(0) - b.mean(0))
        pooled = np.sqrt(
            (a.var(0, ddof=1).mean() + b.var(0, ddof=1).mean()) / 2
        )

        pair_rows.append([
            c1,
            c2,
            centre,
            pooled,
            centre / pooled
        ])

sep = pd.DataFrame(
    pair_rows,
    columns=[
        "class_1",
        "class_2",
        "centroid_distance",
        "pooled_feature_sd",
        "standardised_centroid_separation"
    ]
).sort_values("standardised_centroid_separation")

print(
    "\nPairwise class separation: centroid distance divided by "
    "pooled within-class feature SD"
)
display(sep.round(4))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=sep,
    x="standardised_centroid_separation",
    y=sep.class_1 + " vs " + sep.class_2
)
plt.title("Pairwise DGA class separation")
plt.tight_layout()
plt.show()

## 10. Closest cross-label observations

The closest observations with different target labels provide concrete examples of target overlap in the observed DGA feature space.

These examples are qualitative evidence only and are not themselves model errors.

In [ ]:
# 10. The nearest cross-label observation pairs offer concrete report-ready failure examples.

cross = np.where(~same)[0]
order = np.argsort(pair_distance[cross])[:15]

records = []

for pos in cross[order]:
    i, j = iu[0][pos], iu[1][pos]

    r = {
        "row_a": int(df.index[i]),
        "class_a": df[T].iloc[i],
        "row_b": int(df.index[j]),
        "class_b": df[T].iloc[j],
        "standardised_distance": pair_distance[pos]
    }

    for f in raw:
        r[f] = round((df[f].iloc[i] + df[f].iloc[j]) / 2, 3)

    records.append(r)

close_cross = pd.DataFrame(records)

print(
    "\nFifteen closest cross-label pairs. Gas columns are pair means "
    "for compact display; inspect original row IDs if needed."
)
display(close_cross.round(4))

## 11. V2.5 local-signal gate and conclusion

The diagnostic gate requires both 1-neighbour and 5-neighbour purity to exceed the permutation 95th percentile with a small empirical one-sided p-value.

A failure of this gate means that further model tuning should not be treated as the immediate priority; target construction and the information available in the supplied features should be reviewed first.

In [ ]:
# 11. Decision is deliberately a diagnostic checkpoint, not a route to tuning.

p1 = purity_table.loc[
    purity_table.metric == "1-neighbour purity"
].iloc[0]

p5 = purity_table.loc[
    purity_table.metric == "5-neighbour purity"
].iloc[0]

local_signal = (
    p1.observed > p1.permutation_95th_pct
    and p1.empirical_one_sided_p < .05
    and p5.observed > p5.permutation_95th_pct
    and p5.empirical_one_sided_p < .05
)

if local_signal:
    verdict = (
        "LOCAL SIGNAL DETECTED: nearest-neighbour consistency is clearly "
        "above the permutation control. This supports a carefully isolated "
        "V3 study, but does not require it."
    )
else:
    verdict = (
        "LOCAL SIGNAL NOT ESTABLISHED: nearest DGA neighbours are not more "
        "label-coherent than the permutation control. Prioritise "
        "target-generation review before V3 tuning."
    )

gate = pd.DataFrame([
    [
        "1-neighbour purity exceeds null 95th percentile",
        p1.observed > p1.permutation_95th_pct,
        f"{p1.observed:.4f} vs {p1.permutation_95th_pct:.4f}; "
        f"p={p1.empirical_one_sided_p:.4f}"
    ],
    [
        "5-neighbour purity exceeds null 95th percentile",
        p5.observed > p5.permutation_95th_pct,
        f"{p5.observed:.4f} vs {p5.permutation_95th_pct:.4f}"
    ],
    [
        "Same-class distances are lower than different-class distances",
        p < .05,
        f"Mann-Whitney p={p:.4g}"
    ]
], columns=["diagnostic criterion", "pass", "evidence"])

print("\nV2.5 local-signal gate")
display(gate)

print("=" * 78)
print("V2.5 CONCLUSION")
print(verdict)

print(
    "Provider question: Was fault_by_rogers generated directly from supplied "
    "DGA values/ratios, from hidden inputs, or independently? Please provide "
    "the target-generation rule and any withheld variables."
)

---
# V3 — Final Fixed-Model Error Analysis

**Purpose**

V3 characterises how the fixed V1/V2 XGBoost model fails on held-out observations.

The analysis uses repeated out-of-fold predictions and reports aggregate performance, class-specific precision/recall/F1, count and row-normalised confusion matrices, probability-confidence diagnostics, closest cross-label DGA pairs, and high-confidence incorrect predictions.

No hyperparameter tuning is performed.

The structure below follows the same style as V1: environment setup → imports and reproducibility → dataset loading → numbered analytical sections → explicit conclusion.

In [ ]:
# 2. Environment setup

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "xgboost": "xgboost",
}

missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Environment ready.")

In [ ]:
# 3. Imports and reproducibility

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist, squareform

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
np.random.seed(RANDOM_STATE)

print("Random state:", RANDOM_STATE)
print(f"Repeated CV: {N_SPLITS} folds x {N_REPEATS} repeats")

## 4. Load the dataset

The same `dummy_dga_dataset.xlsx` used in V1 is loaded so that the validation stages operate on the identical observations and target definition.

In [ ]:
df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 5. Fixed XGBoost model and repeated out-of-fold evaluation

The exact fixed V1/V2 XGBoost configuration is retained.

Each observation receives one held-out probability vector in each repeat. The five independent out-of-fold probability vectors are averaged to obtain the final descriptive OOF prediction for that observation.

In [ ]:
T = "fault_by_rogers"
raw = ["H2", "C2H2", "C2H4", "C2H6", "CH4", "CO", "CO2"]
ratios = ["R_C2H2_C2H4", "R_CH4_H2", "R_C2H4_C2H6"]
features = raw + ratios

encoder = LabelEncoder()
y = encoder.fit_transform(df[T])
classes = list(encoder.classes_)
labels = np.arange(len(classes))
X = df[features].copy()

print("V3 final fixed-model evaluation and error analysis")
print(
    "The V1/V2 XGBoost configuration is used without tuning. "
    "Scores are repeated out-of-fold predictions only; asset/time "
    "are not predictive features."
)

def fixed_xgb():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="multi:softprob",
            num_class=len(classes),
            n_estimators=150,
            max_depth=3,
            learning_rate=.05,
            subsample=.9,
            colsample_bytree=.9,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist"
        ))
    ])

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

prob_sum = np.zeros((len(df), len(classes)))
n_oof = np.zeros(len(df), dtype=int)
fold_rows = []

for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
    m = fixed_xgb().fit(X.iloc[tr], y[tr])
    p = m.predict_proba(X.iloc[te])
    pr = p.argmax(axis=1)

    prob_sum[te] += p
    n_oof[te] += 1

    fold_rows.append([
        fold,
        f1_score(
            y[te],
            pr,
            average="macro",
            labels=labels,
            zero_division=0
        ),
        accuracy_score(y[te], pr),
        balanced_accuracy_score(y[te], pr)
    ])

assert np.all(
    n_oof == N_REPEATS
), "Each observation must have one OOF prediction per repeat."

oof_prob = prob_sum / n_oof[:, None]
oof_pred = oof_prob.argmax(axis=1)
confidence = oof_prob.max(axis=1)
correct = oof_pred == y

fold_scores = pd.DataFrame(
    fold_rows,
    columns=[
        "validation_fold",
        "macro_f1",
        "accuracy",
        "balanced_accuracy"
    ]
)

summary = pd.DataFrame([[
    "OOF repeated-CV aggregate",
    accuracy_score(y, oof_pred),
    balanced_accuracy_score(y, oof_pred),
    f1_score(
        y,
        oof_pred,
        average="macro",
        labels=labels,
        zero_division=0
    ),
    f1_score(
        y,
        oof_pred,
        average="weighted",
        labels=labels,
        zero_division=0
    ),
    fold_scores.macro_f1.mean(),
    fold_scores.macro_f1.std(ddof=1)
]], columns=[
    "evaluation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "mean_fold_macro_f1",
    "fold_macro_f1_sd"
])

print("\nRepeated out-of-fold aggregate performance")
display(summary.round(4))

print(
    "This aggregate combines five held-out probability predictions per "
    "observation. Mean fold macro-F1 remains the primary V2 comparison quantity."
)

plt.figure(figsize=(10, 4))
sns.stripplot(
    data=fold_scores,
    y="macro_f1",
    jitter=.15
)
plt.axhline(
    fold_scores.macro_f1.mean(),
    color="crimson",
    ls="--",
    label="mean"
)
plt.legend()
plt.title("25 fixed-model held-out fold macro-F1 values")
plt.tight_layout()
plt.show()

## 6. Per-class performance and confusion analysis

Overall macro-F1 can hide important differences between fault classes.

This section reports precision, recall and F1 for every class and visualises both the raw confusion counts and the row-normalised confusion matrix.

In [ ]:
# 6. Per-class report and error matrix.

report = pd.DataFrame(
    classification_report(
        y,
        oof_pred,
        target_names=classes,
        output_dict=True,
        zero_division=0
    )
).T

per_class = report.loc[
    classes,
    ["precision", "recall", "f1-score", "support"]
].rename(columns={"f1-score": "f1"})

print("\nPer-class held-out error analysis")
display(per_class.round(4))

cm = confusion_matrix(y, oof_pred, labels=labels)
cm_norm = confusion_matrix(
    y,
    oof_pred,
    labels=labels,
    normalize="true"
)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    ax=ax[0]
)
ax[0].set(
    title="OOF confusion matrix: counts",
    xlabel="Predicted",
    ylabel="Actual"
)

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    ax=ax[1]
)
ax[1].set(
    title="OOF confusion matrix: row-normalised",
    xlabel="Predicted",
    ylabel="Actual"
)

for z in ax:
    z.tick_params(axis="x", rotation=30)
    z.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

errors = []

for i, a in enumerate(classes):
    for j, pred in enumerate(classes):
        if i != j and cm[i, j] > 0:
            errors.append([
                a,
                pred,
                int(cm[i, j]),
                cm_norm[i, j]
            ])

error_table = pd.DataFrame(
    errors,
    columns=[
        "actual_class",
        "predicted_class",
        "count",
        "within_actual_class_rate"
    ]
).sort_values(
    ["count", "within_actual_class_rate"],
    ascending=False
)

print("Most frequent off-diagonal errors")
display(error_table.round(4))

## 7. Probability-confidence diagnostics

The maximum predicted probability is treated descriptively rather than as a tuning target.

The objective is to determine whether the model's probability concentration distinguishes correct from incorrect held-out predictions and whether confident errors occur.

In [ ]:
# 7. Confidence is descriptive: assess whether probability concentration distinguishes correct from incorrect OOF predictions.

result = df[["row_id", "asset", "ts_H2", T]].copy()

result["oof_prediction"] = encoder.inverse_transform(oof_pred)
result["correct"] = correct
result["max_predicted_probability"] = confidence

for i, c in enumerate(classes):
    result[
        "p_" + c.replace(" ", "_").replace("-", "_")
    ] = oof_prob[:, i]

confidence_summary = result.groupby(
    "correct"
)["max_predicted_probability"].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

bins = np.linspace(.25, .70, 6)

result["confidence_bin"] = pd.cut(
    result["max_predicted_probability"],
    bins=bins,
    include_lowest=True
)

confidence_bins = result.groupby(
    "confidence_bin",
    observed=False
).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    mean_confidence=("max_predicted_probability", "mean")
).reset_index()

print("\nProbability-confidence diagnostic")
display(confidence_summary.round(4))
display(confidence_bins.round(4))

plt.figure(figsize=(10, 5))
sns.histplot(
    data=result,
    x="max_predicted_probability",
    hue="correct",
    bins=18,
    stat="density",
    common_norm=False,
    element="step"
)
plt.title(
    "OOF maximum predicted probability: correct vs incorrect predictions"
)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(
    data=confidence_bins,
    x="confidence_bin",
    y="accuracy"
)
plt.ylim(0, 1)
plt.xticks(rotation=25)
plt.title("Observed OOF accuracy by confidence bin")
plt.tight_layout()
plt.show()

# Multiclass Brier score, averaged across one-vs-all targets.
brier = np.mean([
    brier_score_loss(
        (y == i).astype(int),
        oof_prob[:, i]
    )
    for i in labels
])

print(
    f"Mean one-vs-rest Brier score: {brier:.4f} "
    "(descriptive probability-quality measure; not a tuning target)."
)

## 8. Closest cross-label DGA pairs

The V2.5 feature geometry is reused here to connect the target-overlap evidence with the model's error analysis.

These pairs are qualitative failure evidence only. A pair does not need to be misclassified to demonstrate that different target labels can occupy very similar observed DGA feature space.

In [ ]:
# 8. V2.5 qualitative evidence: closest pairs with different labels in exactly the same standardised feature space.

Z = StandardScaler().fit_transform(
    SimpleImputer(strategy="median").fit_transform(X)
)

D = squareform(pdist(Z))
iu = np.triu_indices(len(df), 1)

cross = y[iu[0]] != y[iu[1]]
positions = np.where(cross)[0]

close = positions[
    np.argsort(D[iu][positions])[:15]
]

pair_rows = []

for pos in close:
    i, j = iu[0][pos], iu[1][pos]

    pair_rows.append([
        int(df.row_id.iloc[i]),
        df[T].iloc[i],
        result.oof_prediction.iloc[i],
        round(float(confidence[i]), 3),
        int(df.row_id.iloc[j]),
        df[T].iloc[j],
        result.oof_prediction.iloc[j],
        round(float(confidence[j]), 3),
        round(float(D[i, j]), 4)
    ])

close_pairs = pd.DataFrame(
    pair_rows,
    columns=[
        "row_id_a",
        "actual_a",
        "OOF_pred_a",
        "confidence_a",
        "row_id_b",
        "actual_b",
        "OOF_pred_b",
        "confidence_b",
        "standardised_DGA_distance"
    ]
)

print("\nClosest cross-label DGA pairs from V2.5 feature geometry")
display(close_pairs)

print(
    "These are qualitative failure evidence only: a pair may not itself "
    "be misclassified, but close different labels demonstrate target overlap "
    "in observed feature space."
)

## 9. Most confident incorrect predictions

High-confidence errors are particularly important when the observed feature representation is weak.

The following table lists the incorrect OOF predictions with the highest maximum predicted probability.

In [ ]:
# 9. Representative high-confidence errors are particularly important when the representation is weak.

wrong = result.loc[
    ~result.correct,
    [
        "row_id",
        "asset",
        "ts_H2",
        T,
        "oof_prediction",
        "max_predicted_probability"
    ]
].sort_values(
    "max_predicted_probability",
    ascending=False
).head(15)

print("\nFifteen most confident incorrect OOF predictions")
display(wrong.round(4))

## 10. V3 final error-analysis conclusion

The final interpretation is deliberately tied to the earlier validation stages.

V3 characterises failure modes of the fixed model. It does **not** convert the observed performance into justification for hyperparameter tuning.

In [ ]:
print("=" * 78)
print("V3 FINAL ERROR-ANALYSIS CONCLUSION")
print(
    "The fixed XGBoost model produces substantial, class-specific held-out "
    "confusion. Its probability concentration and high-confidence errors "
    "should be interpreted alongside V2's failed permutation gate and "
    "V2.5's absent local label consistency. These results characterise "
    "failure; they do not justify tuning."
)

---
# Final Investigation Summary

The notebook now presents the investigation in the same structured style throughout:

- **V1:** establish whether the supplied data shows initial evidence of learnable DGA-to-fault structure.
- **V2:** test whether that evidence survives stronger validation, permutation controls, representation checks, asset separation and temporal validation.
- **V2.5:** directly test whether the target labels are locally coherent in the supplied DGA feature space.
- **V3:** characterise the fixed model's held-out errors without tuning.

The version boundaries are intentionally explicit so that the notebook can be used as the implementation record for the assignment and as the basis for explaining the research progression during the presentation/Q&A.